# Data Preprocessing

In this file, we will perform Data Preparation before using it to train Machine Learning models.
The main steps include:
1. **Load Data**: Load raw data
2. **Target Variable**: Create and filter target (Turnover Intention)
3. **Missing Values**: Handle missing values and 'X' responses in the survey (Don't Know / No Basis to Judge)
4. **Feature Selection**: Select columns to be used as predictors
5. **Save Processed Data**: Save as a CSV file for the model training process

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

## 1. Load Data & Define Target Variable

In [2]:
# Load raw dataset
data_path = '../data/FEVS2022_PRDF_CSV/2022_OPM_FEVS_PRDF.csv'
df = pd.read_csv(data_path, low_memory=False)

# Create 'Turnover_Intention' column and drop those who answered Retire (2) or did not answer
df['Q93'] = pd.to_numeric(df['Q93'], errors='coerce')
df = df.dropna(subset=['Q93'])
df = df[df['Q93'] != 2]

# 0 = Stay (answered 1), 1 = Leave (answered 3,4,5,6)
df['Turnover_Intention'] = df['Q93'].apply(lambda x: 1 if x in [3, 4, 5, 6] else 0)

print(f"Data shape after filtering target: {df.shape}")
print(df['Turnover_Intention'].value_counts(normalize=True))

Data shape after filtering target: (501974, 117)
Turnover_Intention
0    0.801565
1    0.198435
Name: proportion, dtype: float64


## 2. Feature Selection & Missing Values

Select assessment questions `Q1` to `Q100` and some demographic variables.
Replace 'X' (No opinion/Don't know) with `NaN` (Missing Value) so the model can distinguish or impute later.

In [3]:
# Select question columns Q1 - Q100 (if any)
q_cols = [col for col in df.columns if col.startswith('Q') and col != 'Q93']

# [Revised] Select Demographic columns actually present in the OPM FEVS 2022 dataset
demographic_cols = ['DSEX', 'DAGEGRP', 'DSUPER', 'DFEDTEN', 'DRNO', 'DHISP', 'DDIS', 'DMIL']

# Combine into Features to use
selected_features = q_cols + demographic_cols

# Subset data to include only target column, predictors, and Sample Weight
df_selected = df[selected_features + ['Turnover_Intention', 'POSTWT']].copy()

# Ensure POSTWT is numeric
df_selected['POSTWT'] = pd.to_numeric(df_selected['POSTWT'], errors='coerce')

# Replace 'X' and other blanks with np.nan for predictor columns
df_selected[selected_features] = df_selected[selected_features].replace('X', np.nan)
df_selected[selected_features] = df_selected[selected_features].replace(' ', np.nan)

# Convert all features to Numeric if possible
for col in selected_features:
    df_selected[col] = pd.to_numeric(df_selected[col], errors='coerce')

print("Missing Values Summary (%):")
missing_percent = df_selected.isnull().mean() * 100
print(missing_percent[missing_percent > 0].sort_values(ascending=False).head(10))

Missing Values Summary (%):
DMIL       100.000000
DFEDTEN    100.000000
DSEX       100.000000
DAGEGRP    100.000000
DSUPER     100.000000
DDIS       100.000000
DHISP      100.000000
DRNO       100.000000
Q83         43.005415
Q84         42.184854
dtype: float64


In this case, when using Tree-based models (Such as XGBoost, LightGBM), we can leave `NaN` as is since the model can handle missing values automatically.

## 3. Save Processed Data
We will save the subsetted data with 'X' replaced by `NaN` into a new file.

In [4]:
# Save into Folder data/(new file name)
output_path = '../data/fevs_processed_for_ml.csv'
df_selected.to_csv(output_path, index=False)
print(f"Saved processed dataset to {output_path} with shape {df_selected.shape}")

Saved processed dataset to ../data/fevs_processed_for_ml.csv with shape (501974, 113)
